# 06 · AHR Go player with a measured thinking-time budget

**Purpose:** connect the native Queen/Worker/Scout model from 04 and the shared representation from 05 to complete Go states. Learned role energies guide Scout proposals and retained moves, Workers apply authoritative Go transitions, and Queen governance enforces budgets and independently verified emission.

**Inputs:** only tagged definition cells from 00, 01, 04, and 05; a supplied frozen `NativeAHR` model or the explicitly untrained fixture below. No training, external playing engine, development dataset, or final/lockbox dataset is used.

**Outputs:** callable `choose_ahr_go_move`, typed Go governance and independent final verifier, `reports/06_go_search_validation.json`, `config/06_go_search.json`. Expected **under 2 minutes** for Run All on CPU. Validation calls are bounded to seconds; this notebook starts no long run.

**Run All:** select **AHR Go (.venv)**, then Restart Kernel and Run All. Run from the workspace or `notebooks/`. Interruption recovery: restart the kernel and Run All to repeat these short, stateless, forward-only checks. Inference does not update weights, optimizer state, sampler, or datasets. No long-lived search is checkpointed here; callers must checkpoint between moves before building a long match workflow.

**Interpretation:** the supplied AHR theory permits finite energy ranking under hard feasibility, but does not supply a proved Go value function. This implementation uses alternating adversarial backup of learned trajectory preferences over a bounded retained tree. Depth and branch limits are heuristic deferrals, never proved pruning. Every strategic result remains **UNKNOWN**. A verified legal move establishes legality only. Fresh weights provide a functional player, not demonstrated Go skill.

Timing is cooperative: deadlines are checked around every state transition and model microstep, and elapsed time includes final independent verification. An in-flight Python/GPU operation cannot be preempted; reported overruns remain visible. Zero budget returns a verified pass on a nonterminal state. No hard real-time guarantee is claimed.

In [ ]:
from pathlib import Path
import os, json, hashlib, time, math, copy
import nbformat

PROJECT = Path(os.environ.get("AHR_GO_PROJECT", Path.cwd())).resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
NOTEBOOK_PATH = PROJECT / "notebooks/06_ahr_go_search_engine.ipynb"
SOURCE_HASHES = {}
for filename in ("00_instance_setup_and_recovery.ipynb", "01_go_rules_and_independent_verifier.ipynb",
                 "04_native_ahr_architecture.ipynb", "05_csp_qcsp_pretraining.ipynb"):
    notebook = nbformat.read(PROJECT / "notebooks" / filename, as_version=4)
    sources = [cell.source for cell in notebook.cells
               if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags", [])]
    if not sources:
        raise RuntimeError(f"No definition exports in {filename}")
    for index, source in enumerate(sources):
        exec(compile(source, f"{filename}:ahr-export-{index}", "exec"), globals())
    SOURCE_HASHES[filename] = hashlib.sha256("\n".join(sources).encode()).hexdigest()
print("Imported definitions only; upstream workflows were not executed.")

## Complete state and independent verification

The search keeps the full immutable `GoState`, including positional-superko history, consecutive passes, move count, turn, and komi. The codec also receives this complete state; overflow causes abstention with a legal fallback instead of silently dropping history. Neural masking is reconstructed from rule-checked transitions and cannot authorize an illegal move.

`independent_go_successor` below uses a coordinate graph with flood-fill groups and independently constructs the complete successor. It does not call `play`, `group_and_liberties`, or the neural model. Final acceptance compares its result with the Worker's authoritative successor, including history and counters.

In [ ]:
from dataclasses import dataclass, asdict, replace
from typing import Optional

GO_SEARCH_VERSION = "ahr-go-energy-tree-v1"


def go_state_digest(state):
    return hashlib.sha256(json.dumps(state.to_dict(), sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def independent_go_successor(state, action):
    """Independent one-move certificate checker; None means illegal, never a loss."""
    if type(action) is not int or not 0 <= action <= state.size ** 2 or state.terminal:
        return None
    n = state.size
    if action == n * n:
        values = state.board
        passes = state.consecutive_passes + 1
        history = state.history
    else:
        location = divmod(action, n)
        stones = {(r, c): state.board[r * n + c] for r in range(n) for c in range(n)}
        if stones[location] != 0:
            return None
        def neighbors(point):
            r, c = point
            return [(rr, cc) for rr, cc in ((r-1, c), (r+1, c), (r, c-1), (r, c+1))
                    if 0 <= rr < n and 0 <= cc < n]
        def connected(seed):
            color, seen, queue, freedoms = stones[seed], {seed}, [seed], set()
            while queue:
                current = queue.pop()
                for adjacent in neighbors(current):
                    if stones[adjacent] == 0:
                        freedoms.add(adjacent)
                    elif stones[adjacent] == color and adjacent not in seen:
                        seen.add(adjacent)
                        queue.append(adjacent)
            return seen, freedoms
        stones[location] = state.to_play
        visited = set()
        for adjacent in neighbors(location):
            if stones[adjacent] == -state.to_play and adjacent not in visited:
                group, liberties = connected(adjacent)
                visited.update(group)
                if not liberties:
                    for point in group:
                        stones[point] = 0
        if not connected(location)[1]:
            return None
        values = tuple(stones[(r, c)] for r in range(n) for c in range(n))
        if values in state.history:
            return None
        history, passes = state.history | {values}, 0
    return GoState(size=n, board=values, to_play=-state.to_play, komi=state.komi,
                   history=history, consecutive_passes=passes, move_number=state.move_number+1)


def verify_go_emission(parent, action, proposed_child):
    checked = independent_go_successor(parent, action)
    return (checked is not None and isinstance(proposed_child, GoState)
            and checked.to_dict() == proposed_child.to_dict())


@dataclass(frozen=True)
class AHRGoSearchConfig:
    max_depth: int = 2
    branch_width: int = 3
    max_nodes: int = 64
    queen_cycles: int = 1
    score_discount: float = 0.9
    audit_reserve_s: float = 0.002

    def __post_init__(self):
        for name in ("max_depth", "branch_width", "max_nodes", "queen_cycles"):
            if type(getattr(self, name)) is not int or getattr(self, name) < 1:
                raise ValueError(f"{name} must be a positive integer")
        if self.max_depth > 16 or self.max_nodes > 100000:
            raise ValueError("Use bounded depth <=16 and node count <=100000")
        if isinstance(self.score_discount, bool) or not math.isfinite(self.score_discount) or not 0 < self.score_discount <= 1:
            raise ValueError("score_discount must be finite in (0,1]")
        if isinstance(self.audit_reserve_s, bool) or not math.isfinite(self.audit_reserve_s) or self.audit_reserve_s < 0:
            raise ValueError("audit_reserve_s must be finite and nonnegative")


@dataclass(frozen=True)
class GoQueenGovernance:
    root_color: int
    deadline: float
    node_limit: int
    depth_limit: int
    allowed_goal: str = "choose_legal_move"
    verification_required: bool = True
    strategy: str = "ahr_energy_adversarial_backup"


@dataclass(frozen=True)
class GoWorkerConstruction:
    state: GoState
    path: tuple[int, ...] = ()
    active_goal: str = "choose_legal_move"
    rule: str = "authoritative_go_transition"


@dataclass(frozen=True)
class GoScoutFrontier:
    proposed: tuple[int, ...]
    retained: tuple[int, ...]
    deferred: tuple[int, ...]
    role_energies: tuple[tuple[int, float, float, float, float], ...]


def go_qws_clauses(queen, worker, scout, action, *, verified=False, decision="extend", stop=False,
                   exhausted=False):
    """Six hard compatibility clauses, with no learned proof or safe-pruning claim."""
    relevant = frozenset(range(worker.state.pass_action + 1)) if not worker.state.terminal else frozenset()
    candidate = None
    if type(action) is int and action in relevant:
        try:
            candidate = play(worker.state, action)
        except IllegalMove:
            pass
    legal = candidate is not None
    # Deferred legal actions remain unexamined; none are labelled infeasible.
    return {
        "QWS-1": worker.active_goal == queen.allowed_goal == "choose_legal_move",
        "QWS-2": set(scout.proposed) <= relevant and set(scout.retained) <= set(scout.proposed)
                     and set(scout.deferred) <= set(scout.proposed)
                     and not (set(scout.retained) & set(scout.deferred)),
        "QWS-3": action in scout.retained and legal and worker.rule == "authoritative_go_transition",
        "QWS-4": decision in ("extend", "accept", "abstain")
                     and (decision != "accept" or legal),
        "QWS-5": not (queen.verification_required and decision == "accept") or verified is True,
        "QWS-6": not stop or (decision == "accept" and verified is True) or (decision == "abstain" and exhausted),
    }


def go_worker_transition(queen, worker, scout, action):
    clauses = go_qws_clauses(queen, worker, scout, action)
    if not all(clauses.values()):
        raise ValueError("Incompatible Go QWS transition: " + ", ".join(k for k,v in clauses.items() if not v))
    child = play(worker.state, action)
    return GoWorkerConstruction(child, worker.path + (action,))


## Learned AHR energies govern the retained tree

Every node supplies board/history/task context and all board/pass action payloads to the shared 05 codec. The authoritative legality mask replaces the candidate mask before the NativeAHR call. Queen, Worker, Scout and pheromone energies jointly order the actions; the lowest-energy actions form the retained frontier. NativeAHR completes its nested recurrence in individually timed Scout microsteps, keeping pheromone and role states between those microsteps. Different Go positions start new recurrent tasks, because 04 binds recurrence to its exact input.

A branch contributes `−tanh(total_energy)` in its mover's orientation. The root-oriented discounted sum is a **learned preference heuristic**, not a trained or calibrated value prediction. Opponent nodes minimize root utility, root-side nodes maximize it; true terminal states use the rules' final area score. Only explored, retained replies participate. All other replies remain unknown. Beam/depth/node/deadline limits can miss the best move or a refutation.

In [ ]:
class GoSearchCutoff(RuntimeError):
    pass


def _go_sync(model):
    device = next(model.parameters()).device
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def _go_check_deadline(deadline):
    if time.monotonic() >= deadline:
        raise GoSearchCutoff("deadline")


def _go_rank_actions(state, root_color, depth, model, codec_config, queen, search_config, counters):
    _go_check_deadline(queen.deadline)
    actions = list(range(state.pass_action + 1))
    legal = []
    for action in actions:
        _go_check_deadline(queen.deadline)
        try:
            play(state, action)
        except IllegalMove:
            continue
        legal.append(action)
    if not legal:
        raise GoSearchCutoff("no_legal_candidate")
    context = {"task": "go", "state": state.to_dict(), "root_color": root_color,
               "search_depth": depth, "rules": RULESET["name"]}
    reference = next(model.parameters())
    try:
        batch = encode_reasoning_actions(context, [{"action": a} for a in actions], codec_config,
                                         device=str(reference.device))
    except ValueError as error:
        raise GoSearchCutoff("codec_capacity_or_invalid_context: " + str(error)) from error
    if not isinstance(batch, dict) or set(batch) != {"x", "valid_mask", "action_mask"}:
        raise ValueError("05 codec must return exactly x, valid_mask, action_mask")
    indices = batch["action_mask"][0].nonzero(as_tuple=False).flatten()
    if indices.numel() != len(actions):
        raise ValueError("05 codec candidate anchors must preserve one index per supplied action")
    batch["x"] = batch["x"].to(dtype=reference.dtype)
    allowed = torch.zeros_like(batch["action_mask"])
    for action in legal:
        allowed[0, indices[action]] = True
    batch["action_mask"] = allowed
    recurrent, output = None, None
    microsteps = search_config.queen_cycles * model.config.worker_steps * model.config.scout_steps
    for _ in range(microsteps):
        _go_check_deadline(queen.deadline)
        output = model(batch, micro_steps=1, state=recurrent)
        _go_sync(model)
        counters["model_microsteps"] += 1
        recurrent = output["state"]
        _go_check_deadline(queen.deadline)
    counters["model_calls"] += 1
    rows = []
    for action in legal:
        index = indices[action]
        energies = tuple(float(output["energies"][name][0, index].detach().cpu())
                         for name in ("queen", "worker", "scout", "pheromone"))
        if not all(math.isfinite(value) for value in energies):
            raise FloatingPointError("Nonfinite learned action energy")
        rows.append((action, *energies))
    rows.sort(key=lambda row: (sum(row[1:]), row[0]))
    retained = tuple(row[0] for row in rows[:search_config.branch_width])
    return GoScoutFrontier(tuple(legal), retained,
                           tuple(row[0] for row in rows[search_config.branch_width:]), tuple(rows))


def choose_ahr_go_move(state, model, codec_config, time_budget_s=1.0, search_config=None):
    """Frozen native AHR player; legal output, UNKNOWN strategy, observable overrun.

    Caller must provide a frozen/eval NativeAHR instance and 05 codec config.
    This callable does not load, train, or mutate model weights. Terminal input
    returns no action. Search is deliberately stateless between turns.
    """
    start = time.monotonic()
    if not isinstance(state, GoState):
        raise TypeError("state must be the complete GoState from notebook 01")
    if isinstance(time_budget_s, bool) or not isinstance(time_budget_s, (int,float)) or not math.isfinite(time_budget_s) or time_budget_s < 0:
        raise ValueError("time_budget_s must be finite and nonnegative")
    if not isinstance(model, NativeAHR) or model.training:
        raise ValueError("Pass an eval-mode NativeAHR; training is never started by this player")
    config = AHRGoSearchConfig() if search_config is None else search_config
    if not isinstance(config, AHRGoSearchConfig):
        raise TypeError("search_config must be AHRGoSearchConfig")
    if config.queen_cycles > model.config.max_iterations:
        raise ValueError("Queen cycle request exceeds the architecture bound")
    queen = GoQueenGovernance(state.to_play,
        start + max(0.0, time_budget_s - config.audit_reserve_s), config.max_nodes, config.max_depth)
    counters = {"nodes": 0, "model_calls": 0, "model_microsteps": 0,
                "deferred_actions": 0, "depth_cutoffs": 0, "deepest_ply": 0,
                "opponent_nodes": 0}
    if state.terminal:
        elapsed = time.monotonic() - start
        return {"action": None, "status": "TERMINAL", "legality_verified": False,
                "strategic_status": "UNKNOWN", "reason": "game_ended_after_two_passes",
                "elapsed_s": elapsed, "time_budget_s": time_budget_s,
                "deadline_exceeded": elapsed > time_budget_s, "counters": counters,
                "root_candidates": [], "model_weights_changed": False}
    root_worker = GoWorkerConstruction(state)
    selected, selected_value = state.pass_action, None
    root_candidates, traces = [], []
    reason = "bounded_search_complete"
    fallback = True
    def explore(worker, depth, path_value):
        _go_check_deadline(queen.deadline)
        counters["deepest_ply"] = max(counters["deepest_ply"], depth)
        if worker.state.terminal:
            return math.tanh(area_score(worker.state) * queen.root_color / (state.size ** 2))
        if depth >= config.max_depth:
            counters["depth_cutoffs"] += 1
            return path_value
        if counters["nodes"] >= queen.node_limit:
            raise GoSearchCutoff("node_budget")
        counters["nodes"] += 1
        if worker.state.to_play != queen.root_color:
            counters["opponent_nodes"] += 1
        frontier = _go_rank_actions(worker.state, queen.root_color, depth, model, codec_config,
                                    queen, config, counters)
        counters["deferred_actions"] += len(frontier.deferred)
        energy_map = {row[0]: sum(row[1:]) for row in frontier.role_energies}
        values = []
        for action in frontier.retained:
            _go_check_deadline(queen.deadline)
            next_worker = go_worker_transition(queen, worker, frontier, action)
            _go_check_deadline(queen.deadline)
            mover_sign = 1.0 if worker.state.to_play == queen.root_color else -1.0
            contribution = mover_sign * -math.tanh(energy_map[action])
            updated_value = path_value + config.score_discount ** depth * contribution
            value = explore(next_worker, depth + 1, updated_value)
            values.append((value, action))
            traces.append({"path": list(next_worker.path), "depth": depth+1,
                           "soft_energy": energy_map[action], "root_heuristic": value})
            if depth == 0:
                root_candidates.append({"action": action, "value": value,
                                        "soft_energy": energy_map[action]})
        if not values:
            raise GoSearchCutoff("empty_retained_frontier")
        return (max(value for value, action in values) if worker.state.to_play == queen.root_color
                else min(value for value, action in values))
    try:
        with torch.inference_mode():
            explore(root_worker, 0, 0.0)
    except GoSearchCutoff as error:
        reason = str(error)
    if root_candidates:
        best = max(root_candidates, key=lambda item: (item["value"], -item["soft_energy"], -item["action"]))
        selected, selected_value, fallback = best["action"], best["value"], False
    # No neural callback contributes a certificate. Re-run both implementations.
    proposed_child = play(state, selected)
    verified = verify_go_emission(state, selected, proposed_child)
    if not verified:
        raise RuntimeError("Independent final legal-move verification failed; no move emitted")
    acceptance_frontier = GoScoutFrontier((selected,), (selected,), (), ())
    clauses = go_qws_clauses(queen, root_worker, acceptance_frontier, selected,
                            verified=verified, decision="accept", stop=True)
    if not all(clauses.values()):
        raise RuntimeError("Final QWS acceptance failed")
    state_identity = go_state_digest(state)
    successor_identity = go_state_digest(proposed_child)
    search_configuration = asdict(config)
    elapsed = time.monotonic() - start
    exceeded = elapsed > time_budget_s
    if exceeded:
        reason = reason + "; deadline_exceeded_including_final_audit"
    return {"action": selected, "status": "LEGAL", "legality_verified": True,
            "strategic_status": "UNKNOWN", "reason": reason,
            "fallback_pass": fallback, "heuristic_value": selected_value,
            "elapsed_s": elapsed, "time_budget_s": time_budget_s,
            "deadline_exceeded": exceeded, "overrun_s": max(0.0, elapsed-time_budget_s),
            "counters": counters, "root_candidates": root_candidates, "trace": traces,
            "qws_clauses": clauses, "state_digest": state_identity,
            "successor_digest": successor_identity,
            "model_weights_changed": False, "search_config": search_configuration}


## Forward-only smoke validation

The following fixture is explicitly **untrained**. Validation exercises a real opponent ply, source-state preservation, ordinary captures, suicide, superko, two-pass termination, all six governance checks, exact hard masks, invalid budgets, and an expired budget. It also verifies that role-energy changes alter branch ordering without treating that sensitivity as playing strength. No optimizer is created and no dataset is opened.

The CPU checks use ample budgets to avoid speed-dependent correctness assertions. The latency sample reports measured timing separately. Capacity overflow is a recorded `UNKNOWN` fallback; it is not converted to a loss or silently shortened input.

In [ ]:
torch.set_num_threads(1)
torch.manual_seed(20260923)
CODEC = ReasoningCodecConfig()
MODEL_CONFIG = AHRConfig(input_dim=16, hidden_dim=32, iterations=1, worker_steps=2,
                         scout_steps=2, max_iterations=4, max_tokens=CODEC.max_tokens, dropout=0.0)
PLAYER = NativeAHR(MODEL_CONFIG).eval()
WEIGHTS_BEFORE = {key: value.detach().clone() for key, value in PLAYER.state_dict().items()}
SEARCH = AHRGoSearchConfig(max_depth=2, branch_width=2, max_nodes=16, queen_cycles=1)
CONFIGURATION = {"version": GO_SEARCH_VERSION, "search": asdict(SEARCH),
                 "model": asdict(MODEL_CONFIG), "codec": asdict(CODEC),
                 "training_steps": 0, "weights": "random_initialization"}
checks = {}
print("Forward-only fixture; optimizer updates = 0")

In [ ]:
# Independent rules comparison on representative Go positions, including explicit history.
fixtures = [initial_state(3, komi=0.5), initial_state(5, komi=0.5)]
occupied = play(initial_state(3, komi=0.5), 4)
fixtures.append(occupied)
suicide = GoState(3, (0,-1,0,-1,0,-1,0,-1,0), to_play=1, komi=0.5)
fixtures.append(suicide)
capture = GoState(3, (0,1,0,1,-1,0,0,1,0), to_play=1, komi=0.5)
fixtures.append(capture)
base = initial_state(3, komi=0.5)
repeated_board = play(base, 0).board
superko = GoState(3, base.board, to_play=1, komi=0.5, history={base.board, repeated_board})
fixtures.append(superko)
fixtures.extend([play(base, base.pass_action), play(play(base, base.pass_action), base.pass_action)])
comparisons = 0
for fixture in fixtures:
    for action in range(-1, fixture.pass_action + 2):
        independent = independent_go_successor(fixture, action)
        try:
            authoritative = play(fixture, action)
        except IllegalMove:
            authoritative = None
        assert (independent is None) == (authoritative is None)
        if authoritative is not None:
            assert independent.to_dict() == authoritative.to_dict()
            assert verify_go_emission(fixture, action, authoritative)
        comparisons += 1
assert independent_go_successor(suicide, 4) is None
assert independent_go_successor(superko, 0) is None
assert play(capture, 5).board[4] == 0
assert independent_go_successor(base, True) is None
checks["independent_rules"] = {"move_comparisons": comparisons, "superko": True,
                               "suicide": True, "capture": True, "pass_terminal": True}

state_before = base.to_dict()
result = choose_ahr_go_move(base, PLAYER, CODEC, 10.0, SEARCH)
assert result["legality_verified"] and result["strategic_status"] == "UNKNOWN"
assert result["counters"]["opponent_nodes"] >= 1 and result["counters"]["deepest_ply"] == 2
assert result["root_candidates"] and not result["fallback_pass"]
assert base.to_dict() == state_before
assert all(result["qws_clauses"].values())
checks["adversarial_two_ply"] = {"chosen_action": result["action"], "counters": result["counters"],
                                  "root_candidates": result["root_candidates"]}
zero = choose_ahr_go_move(base, PLAYER, CODEC, 0.0, SEARCH)
assert zero["action"] == base.pass_action and zero["fallback_pass"]
assert zero["legality_verified"] and zero["strategic_status"] == "UNKNOWN"
assert zero["counters"]["model_microsteps"] == 0
terminal = play(play(base, base.pass_action), base.pass_action)
terminal_result = choose_ahr_go_move(terminal, PLAYER, CODEC, 1.0, SEARCH)
assert terminal_result["action"] is None and terminal_result["status"] == "TERMINAL"
checks["cutoff_semantics"] = {"zero_budget_verified_pass": True, "zero_budget_unknown": True,
                            "terminal_no_move": True}

queen = GoQueenGovernance(1, time.monotonic()+10, 16, 2)
worker = GoWorkerConstruction(base)
frontier = GoScoutFrontier((0,1), (0,), (1,), ())
assert all(go_qws_clauses(queen, worker, frontier, 0).values())
assert not go_qws_clauses(queen, replace(worker, active_goal="other"), frontier, 0)["QWS-1"]
assert not go_qws_clauses(queen, worker, replace(frontier, proposed=(999,)), 0)["QWS-2"]
assert not go_qws_clauses(queen, worker, frontier, 1)["QWS-3"]
assert not go_qws_clauses(queen, worker, frontier, 0, decision="proved_win")["QWS-4"]
assert not go_qws_clauses(queen, worker, frontier, 0, decision="accept")["QWS-5"]
assert not go_qws_clauses(queen, worker, frontier, 0, stop=True)["QWS-6"]
checks["six_qws_guards"] = True

for budget in (True, -1, float("nan"), float("inf")):
    try:
        choose_ahr_go_move(base, PLAYER, CODEC, budget, SEARCH)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid time budget accepted")
checks["invalid_budgets_rejected"] = True
for key, before in WEIGHTS_BEFORE.items():
    assert torch.equal(PLAYER.state_dict()[key], before)
checks["weights_unchanged"] = True
checks["optimizer_updates"] = 0
print("Independent rules, two-ply search, governance, UNKNOWN cutoffs and immutable weights: PASS")

In [ ]:
# The same NativeAHR executes every scoring call. Test-only energy sign reversal
# verifies that its scores, rather than board index or a hidden Go evaluator,
# determine the retained action order.
from unittest.mock import patch
counters_template = {"model_microsteps": 0, "model_calls": 0}
ranking_queen = GoQueenGovernance(1, time.monotonic()+30, 16, 2)
with torch.inference_mode():
    normal = _go_rank_actions(base, 1, 0, PLAYER, CODEC, ranking_queen, SEARCH, dict(counters_template))
original_forward = PLAYER.forward
def reversed_energy_forward(*args, **kwargs):
    output = original_forward(*args, **kwargs)
    output["energies"] = {name: -value for name,value in output["energies"].items()}
    output["policy_logits"] = -(output["energies"]["soft"] + output["hard_action_energy"])
    return output
with patch.object(PLAYER, "forward", side_effect=reversed_energy_forward), torch.inference_mode():
    reversed_scores = _go_rank_actions(base, 1, 0, PLAYER, CODEC, ranking_queen, SEARCH, dict(counters_template))
normal_order = [row[0] for row in normal.role_energies]
reversed_order = [row[0] for row in reversed_scores.role_energies]
assert normal_order != reversed_order
assert normal.retained != reversed_scores.retained
checks["native_energy_controls_frontier"] = {"normal_retained": list(normal.retained),
                                           "reversed_retained": list(reversed_scores.retained)}

mask_checks = []
def inspected_forward(batch, *args, **kwargs):
    output = original_forward(batch, *args, **kwargs)
    # Every disabled token has exactly +infinity hard energy, including
    # candidate anchors removed for positional superko.
    assert torch.isinf(output["hard_action_energy"][~batch["action_mask"]]).all()
    assert (output["policy_logits"].softmax(-1)[~batch["action_mask"]] == 0).all()
    mask_checks.append(int(batch["action_mask"].sum()))
    return output
with patch.object(PLAYER, "forward", side_effect=inspected_forward), torch.inference_mode():
    masked_frontier = _go_rank_actions(superko, 1, 0, PLAYER, CODEC, ranking_queen, SEARCH, dict(counters_template))
assert 0 not in masked_frontier.proposed
assert set(mask_checks) == {len(legal_actions(superko))}
checks["hard_legality_mask"] = {"superko_action_excluded": True, "masked_probability_zero": True}

# Advance a fake clock specifically inside the FINAL independent audit.
# Search has already completed, so the result must still expose this overrun.
clock_now = {"value": 0.0}
original_verifier = verify_go_emission
def delayed_audit(*args, **kwargs):
    outcome = original_verifier(*args, **kwargs)
    clock_now["value"] = 2.0
    return outcome
audit_box = {}
with patch.dict(globals(), {"verify_go_emission": delayed_audit}), patch.object(time, "monotonic", lambda: clock_now["value"]):
    audit_box["result"] = choose_ahr_go_move(base, PLAYER, CODEC, 1.0,
                                 AHRGoSearchConfig(max_depth=1, branch_width=1, max_nodes=2))
audited = audit_box["result"]
assert audited["elapsed_s"] == 2.0 and audited["deadline_exceeded"]
assert audited["strategic_status"] == "UNKNOWN" and audited["legality_verified"]
assert "final_audit" in audited["reason"]
checks["final_audit_in_elapsed_and_deadline"] = True

# A strict representation limit fails visibly and returns a legal UNKNOWN fallback.
small_codec = replace(CODEC, max_tokens=16)
small_result = choose_ahr_go_move(base, PLAYER, small_codec, 1.0, SEARCH)
assert small_result["fallback_pass"] and small_result["legality_verified"]
assert small_result["strategic_status"] == "UNKNOWN" and "codec_capacity" in small_result["reason"]
checks["codec_overflow_never_truncates_state"] = True
print("Learned ranking, hard masking, final-audit timing and representation overflow: PASS")

## Timing and artifact receipt

This is a small runtime sample, not a strength benchmark or a tail-latency guarantee. A later match runner must record every overrun and enforce its own process-level time control if strict deadlines are required. Imported checkpoints remain separate from these random-weight fixtures.

In [ ]:
latencies = []
for size in (3, 5, 9):
    sample = choose_ahr_go_move(initial_state(size), PLAYER, CODEC, 0.25, SEARCH)
    assert sample["legality_verified"] and sample["strategic_status"] == "UNKNOWN"
    latencies.append({"board_size": size, "action": sample["action"], "elapsed_s": sample["elapsed_s"],
                      "deadline_exceeded": sample["deadline_exceeded"], "reason": sample["reason"],
                      "model_microsteps": sample["counters"]["model_microsteps"],
                      "fallback_pass": sample["fallback_pass"]})
checks["latency_samples"] = latencies
for key, before in WEIGHTS_BEFORE.items():
    assert torch.equal(PLAYER.state_dict()[key], before)
notebook = nbformat.read(NOTEBOOK_PATH, as_version=4)
nbformat.validate(notebook)
source_hash = hashlib.sha256("\n".join(cell.source for cell in notebook.cells if cell.cell_type=="code").encode()).hexdigest()
report = {"status": "passed", "version": GO_SEARCH_VERSION, "source_sha256": source_hash,
          "dependency_exports_sha256": SOURCE_HASHES, "checks": checks, "training_steps": 0,
          "model_provenance": "untrained random initialization", "go_training": False,
          "strategic_status": "UNKNOWN", "full_game_optimality_claim": False,
          "timing_contract": "cooperative; elapsed includes final independent audit",
          "final_lockbox_opened": False}
atomic_json(PROJECT / "config/06_go_search.json", CONFIGURATION)
atomic_json(PROJECT / "reports/06_go_search_validation.json", report)
print(json.dumps(report, indent=2))

## Handoff to notebook 07

Import only `ahr-export` cells from the required upstream notebooks. Load the CSP-only checkpoint with 05's inference loader, call `.eval()`, then pass that model and the matching codec configuration to:

```text
choose_ahr_go_move(full_state, frozen_model, codec_config,
                   time_budget_s=1.0,
                   search_config=AHRGoSearchConfig(max_depth=2, branch_width=3))
```

Use the same architecture, codec, search configuration and thinking-time budget for initialized and CSP-trained controls. Save model provenance, legal-action failures, model/search compute, completion status and deadline overruns. A changed preference is not evidence of transfer; 07 must measure that separately on development positions while leaving final/lockbox data sealed.

Storage is the project's persistent boot disk; instance deletion policy remains unverified. This notebook creates no final model checkpoint or off-instance backup.